# Markdown Exploration

This notebook explores the raw markdown structure of the Saudi Building Code (`SBC-201-2007.md`) and validates our Phase A structural parsers.

In [1]:
import sys
from pathlib import Path

# Add root project to the python path
sys.path.append(str(Path.cwd().parent))

from src.preprocessing.section_splitter import ChapterSplitter
from src.preprocessing.md_parser import MarkdownParser


## 1. Load Raw File
Let's load the massive markdown string directly from Phase A's input path.

In [2]:
raw_path = Path.cwd().parent / "data" / "01_raw" / "SBC-201-2007.md"

with open(raw_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"Loaded Source File: {len(content)} characters.")
print("Sample snippet:")
print(content[2400:2600]) # Look near definition scopes

Loaded Source File: 1698576 characters.
Sample snippet:
  |
| 2.11                                          | Utility and Miscellaneous Group U                        |
| 2.12                                          | Special Detailed Requirements Based o


## 2. Test Chapter Splitting
Test how the `ChapterSplitter` performs matching explicit `## CHAPTER X` boundaries.

In [3]:
splitter = ChapterSplitter()
chapters = splitter.split_into_chapters(content)

print(f"Extracted {len(chapters)} blocks.")
for ch in chapters:
    if len(ch['body'].split()) > 10:
        print(f"Chapter {ch['id']}: {ch['title']} | Length: {len(ch['body'])} chars")


Extracted 18 blocks.
Chapter 15: SIGNS | Length: 323 chars
Chapter 16: RODENT PROOFING | Length: 753 chars
Chapter 01: DEFINITIONS | Length: 20439 chars
Chapter 02: USE AND OCCUPANCY CLASSIFICATION | Length: 283308 chars
Chapter 03: GENERAL BUILDING HEIGHTS AND AREAS | Length: 40834 chars
Chapter 04: TYPES OF CONSTRUCTION | Length: 42189 chars
Chapter 05: EXTERIOR WALLS | Length: 42673 chars
Chapter 06: ROOF ASSEMBLIES AND ROOFTOP STRUCTURES | Length: 86076 chars
Chapter 07: INTERIOR ENVIRONMENT | Length: 22845 chars
Chapter 08: MEANS OF EGRESS | Length: 177016 chars
Chapter 09: ACCESSIBILITY | Length: 67570 chars
Chapter 10: GYPSUM BOARD AND PLASTER | Length: 25434 chars
Chapter 11: PLASTIC AND GLASS | Length: 73951 chars
Chapter 12: ENCROACHMENTS INTO THE PUBLIC RIGHT-OF-WAY | Length: 4495 chars
Chapter 13: EXISTING STRUCTURES | Length: 69082 chars
Chapter 14: SAFEGUARDS DURING CONSTRUCTION | Length: 16450 chars
Chapter 15: SIGNS | Length: 18840 chars
Chapter 16: RODENT PROOFING | Le

## 3. Deep Clause Parsing
Using the baseline `MarkdownParser` to map sections inside a generated chapter.

In [4]:
# Grab the first meaningful chapter
sample_chapter = next(ch for ch in chapters if ch["id"] == "01")

md_parser = MarkdownParser()
structure = md_parser.parse_file(sample_chapter['body'])

print(f"Found {len(structure['sections'])} subsections.")
for sec in structure['sections'][:5]:
    print(f"-> Level {sec['level']}: {sec['title']} (Clauses: {len(sec['clauses'])})")


Found 5 subsections.
-> Level 2: CHAPTER 1 DEFINITIONS (Clauses: 0)
-> Level 2: SECTION 1.1 GENERAL (Clauses: 4)
-> Level 2: SECTION 1.2 DEFINITIONS (Clauses: 46)
-> Level 2: CONSTRUCTION TYPES. See Section 4.2 (Clauses: 224)
-> Level 2: STORAGE, HAZARDOUS MATERIALS. See Section 2.27.2 (Clauses: 37)
